# 09. Demostración del Agente Autónomo de Soporte (CrewAI)

Este notebook **no reimplementa** el agente: lo *importa* desde el paquete modular `agent/` y lo ejecuta. De esta forma mantenemos el código modular (requisito **IE7**) y, a la vez, ofrecemos una demostración ejecutable y explicada, consistente con los notebooks 01–08.

**Qué demuestra, mapeado a los Indicadores de Evaluación:**

| Sección | Contenido | IE |
|---------|-----------|----|
| 1 | Herramientas autónomas (validación, correo, búsqueda semántica) | IE2 |
| 2 | Memoria de corto plazo | IE3 |
| 1.3 / 3 | Memoria semántica (FAISS) y ejecución del agente | IE1, IE4, IE5 |
| 4 | Decisiones adaptativas | IE6 |
| 5 | Arquitectura / orquestación | IE8 |

## 0. Requisitos

Antes de ejecutar, desde la **raíz del proyecto** y con el entorno virtual activo:

```bash
pip install -r agent/requirements-agent.txt
```

Y tener configurado el `.env` con `GITHUB_TOKEN` (las secciones 1.3 y 3 lo necesitan; el resto corre sin credenciales). El envío de correo real es opcional (variables `SMTP_*`); sin ellas, el agente usa el **modo simulado**.

## Preparación: hacer visible el paquete `agent/`

El paquete `agent/` vive en la raíz del proyecto (carpeta padre de `notebooks/`). Añadimos esa raíz al `sys.path` para poder importarlo.

In [ ]:
import sys
from pathlib import Path

# Si el notebook se ejecuta desde notebooks/, la raíz es la carpeta padre.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

print("Raíz del proyecto:", ROOT)
print("¿Existe el paquete agent/?:", (ROOT / "agent").exists())

## 1. Herramientas autónomas (Tools) — IE2

El agente decide por sí mismo cuándo invocar cada herramienta. Aquí las probamos de forma aislada para ver su comportamiento.

### 1.1 Validación de usuario  *(funciona sin credenciales)*

In [ ]:
import json
from agent.tools.validation_tool import ValidateUserTool

validar = ValidateUserTool()

# Caso A: faltan datos -> el agente debería pedir información
print("A) Datos incompletos:")
print(json.dumps(json.loads(validar._run(email="correo-malo", issue_description="")), indent=2, ensure_ascii=False))

# Caso B: caso de seguridad válido -> genera ticket y prioridad ALTA
print("\nB) Caso de seguridad válido:")
print(json.dumps(json.loads(validar._run(
    email="jugador@ejemplo.com", steam_id_or_username="jugador77",
    issue_type="seguridad", issue_description="Vi inicios de sesión desde otro país.")),
    indent=2, ensure_ascii=False))

### 1.2 Envío de correo — modo simulado  *(funciona sin credenciales SMTP)*

In [ ]:
from agent.tools.email_tool import SendSupportEmailTool

enviar = SendSupportEmailTool()
resultado = enviar._run(
    to_email="usuario@ejemplo.com",
    subject="Confirmación de tu ticket de soporte",
    body="Hemos recibido tu solicitud y abrimos el ticket correspondiente.",
    category="ticket",
)
print(json.dumps(json.loads(resultado), indent=2, ensure_ascii=False))
print("\nSin credenciales SMTP el correo se guarda como .eml en agent/data/email_outbox/")

### 1.3 Búsqueda semántica en la base de conocimiento  *(requiere `GITHUB_TOKEN`)*

Esta tool consulta la memoria de largo plazo (FAISS). La primera ejecución construye el índice a partir de `agent/data/steam_support_kb.md`.

In [ ]:
from agent.tools.knowledge_tool import KnowledgeBaseSearchTool

try:
    buscar = KnowledgeBaseSearchTool()
    print(buscar._run("¿Cómo recupero mi cuenta si me la robaron?"))
except Exception as e:
    print(f"(Requiere GITHUB_TOKEN configurado y conexión) -> {e}")

## 2. Memoria de corto plazo — IE3  *(funciona sin credenciales)*

Buffer conversacional + estado vivo del caso (ticket, prioridad, validación).

In [ ]:
from agent.memory import ShortTermMemory

mem = ShortTermMemory(window=6)
mem.add_turn("usuario", "No puedo entrar a mi cuenta de Steam")
mem.update_state(ticket_id="STM-123456", priority="ALTA", user_validated=True)
mem.add_turn("agente", "Abrí el ticket STM-123456 con prioridad ALTA")

print("Resumen del estado del caso:")
print(" ", mem.summary())
print("\nHistorial reciente:")
print(mem.transcript())

## 3. Ejecución end-to-end del agente — IE1, IE4, IE5  *(requiere `GITHUB_TOKEN` + crewai)*

`resolver_caso_soporte` ejecuta el plan completo: **validar → buscar en la base de conocimiento → redactar correo → enviar** (con memoria integrada).

In [ ]:
try:
    from agent.crew import resolver_caso_soporte

    resultado = resolver_caso_soporte(
        mensaje_usuario=(
            "Hola, no puedo iniciar sesión y vi un acceso desde otro país. "
            "Creo que me hackearon la cuenta."
        ),
        email_usuario="usuario@ejemplo.com",
        steam_id="jugador_demo_77",
    )
    print("\n=== RESULTADO FINAL DEL CASO ===")
    print(resultado)
except Exception as e:
    print(f"(Requiere instalar agent/requirements-agent.txt y GITHUB_TOKEN) -> {e}")

## 4. Decisiones adaptativas — IE6  *(funciona sin credenciales)*

Demostramos cómo el agente reacciona ante condiciones cambiantes del entorno, usando directamente la lógica de las tools (igual que `tests/test_agent_flows.py`).

In [ ]:
from agent.tools.validation_tool import ValidateUserTool
from agent.tools.email_tool import SendSupportEmailTool
from agent import config

v = ValidateUserTool()

# (a) Chargeback -> escalamiento manual obligatorio
caso_a = json.loads(v._run(email="c@e.com", steam_id_or_username="cli",
        issue_type="facturacion", issue_description="Hice un chargeback por un cargo duplicado."))
print("(a) ¿Escala a humano por chargeback?:", caso_a["requires_manual_escalation"])

# (b) Fallo de envío real -> respaldo simulado + señal de fallo para escalar
config.settings.smtp_user = "demo@example.com"
config.settings.smtp_password = "clave_falsa"
config.settings.smtp_host = "smtp.invalid.local"
e = SendSupportEmailTool(); e.max_retries = 1
caso_b = json.loads(e._run(to_email="u@e.com", subject="x", body="y", category="ticket"))
print("(b) status del envío:", caso_b["status"], "| respaldo:", caso_b["backup"]["status"])
config.settings.smtp_user = ""; config.settings.smtp_password = ""  # restaurar modo simulado

## 5. Arquitectura — diagrama de orquestación (IE8)

```mermaid
flowchart TD
    U([Usuario de Steam]) -->|mensaje + email| ENTRY[resolver_caso_soporte]
    ENTRY --> MEM[Memoria corto + largo plazo]
    ENTRY --> CREW[Crew secuencial]
    subgraph CREW
        T1[1. Validar usuario] --> T2[2. Buscar en KB]
        T2 --> T3[3. Redactar correo] --> T4[4. Enviar correo]
    end
    T1 --> TV[validar_usuario]
    T2 --> TK[buscar_en_base_conocimiento -> FAISS]
    T4 --> TE[enviar_correo_soporte -> SMTP / simulado]
    T4 -->|sent / simulated| OK([Caso cerrado])
    T4 -->|failed| ESC([Escalamiento a humano])
```

El diagrama completo está en `agent/diagrams/orchestration.mermaid` y la documentación del módulo en `agent/README.md`.